# Tenrec QK-article：多路召回（10 万用户）

本 Notebook 面向丹摩智算实例：原始数据、处理中间文件、模型和结果默认写入 `/root/workspace`（数据盘）。它从 QK-article 的 CSV **分块读取**，不会把 4.5GB 原始文件一次性载入内存。

包含四路召回：双塔、ItemCF、UserCF、标签召回；每路输出候选集，并在按原始行顺序切分的留一测试集上计算 Recall@K。Tenrec 移除了时间戳，但论文说明原始行按行为时间顺序排列，因此这里将每位用户的最后一次正反馈作测试、倒数第二次作验证。

## 0. 丹摩路径与执行方式

1. 把 `QK-article.csv` 放到丹摩数据盘，例如 `/root/workspace/data/Tenrec/QK-article.csv`。
2. 只需修改下一格的 `RAW_DATA_PATH`；其余文件会写入 `PROJECT_ROOT`。
3. 推荐单张 4090（24GB 显存）+ 60GB 内存。数据抽样、CF 主要使用 CPU/内存；双塔才使用 GPU。
4. 若要跨实例保留最终结果，可把 `outputs/` 复制到 `/root/shared-storage`。

In [ ]:
# 若镜像缺少依赖，先取消下一行注释并运行一次。
# %pip install -q pandas pyarrow scipy implicit tqdm

from __future__ import annotations

import os
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from implicit.nearest_neighbours import BM25Recommender, CosineRecommender
except ImportError as exc:
    raise ImportError('请先运行上方的 %pip install 单元格。') from exc

SEED = 20260715
N_USERS = 100_000       
MIN_POSITIVE_EVENTS = 7       # 留出验证/测试后，训练至少保留 5 条正反馈
CHUNK_SIZE = 1_000_000
RECALL_K = 100
EVAL_USERS = 2_000            # 先在固定样本上评估，避免全量双塔暴力检索
SIMILAR_K = 100
MAX_HISTORY_FOR_CF = 100
READ_TIME_CAP_SECONDS = 600.0
READ_TIME_WEIGHT = 0.25
READ_PERCENTAGE_WEIGHT = 0.50
# train_only：严格的离线评测；provided_item_stats：使用原始 *_count 统计，仅作特征实验。
POPULARITY_MODE = 'train_only'

# === 丹摩实例中唯一需要按实际位置修改的路径 ===
RAW_DATA_PATH = Path(os.environ.get(
    'TENREC_QK_ARTICLE_PATH',
    '/root/workspace/data/Tenrec/QK-article.csv',
))
PROJECT_ROOT = Path(os.environ.get(
    'TENREC_PROJECT_ROOT',
    '/root/workspace/tenrec_qk_article_multirecall',
))
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
for path in (DATA_DIR, MODEL_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

assert RAW_DATA_PATH.exists(), f'找不到原始文件：{RAW_DATA_PATH}'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'raw data: {RAW_DATA_PATH}')
print(f'project:  {PROJECT_ROOT}')
print(f'device:   {DEVICE}')
print(f'GPU:      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 0.5 原始数据画像（抽样前执行）

这一步完整扫描原始 CSV，但始终分块读取。它统计每列非空数/缺失率、用户和物品基数、正反馈数量、用户与物品交互次数分布，并计算事件级交互密度和稀疏度。报告会保存到 `data/raw_column_profile.csv` 和 `data/raw_profile.json`；再次运行时会直接复用。

In [ ]:
PROFILE_JSON_PATH = DATA_DIR / 'raw_profile.json'
COLUMN_PROFILE_PATH = DATA_DIR / 'raw_column_profile.csv'

def bool_count(series: pd.Series) -> int:
    return int(series.fillna(False).astype(str).str.lower().isin(['1', '1.0', 'true', 't', 'yes']).sum())

if PROFILE_JSON_PATH.exists() and COLUMN_PROFILE_PATH.exists():
    raw_profile = json.loads(PROFILE_JSON_PATH.read_text())
    column_profile = pd.read_csv(COLUMN_PROFILE_PATH)
    print('复用已有原始数据画像。若想重新统计，删除这两个报告文件后重新运行本格。')
else:
    header = pd.read_csv(RAW_DATA_PATH, nrows=0).columns.tolist()
    non_null = {column: 0 for column in header}
    dtypes = {}
    user_counts, item_counts = defaultdict(int), defaultdict(int)
    user_set, item_set = set(), set()
    action_counts = {name: 0 for name in ['click', 'read', 'like', 'share', 'follow', 'favorite']}
    n_rows = 0

    for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, chunksize=CHUNK_SIZE), desc='统计原始数据画像'):
        n_rows += len(chunk)
        for column in header:
            non_null[column] += int(chunk[column].notna().sum())
            dtypes.setdefault(column, str(chunk[column].dtype))

        users = pd.to_numeric(chunk['user_id'], errors='coerce').dropna().astype('int64')
        items = pd.to_numeric(chunk['item_id'], errors='coerce').dropna().astype('int64')
        user_set.update(users.unique().tolist())
        item_set.update(items.unique().tolist())
        for key, value in users.value_counts().items():
            user_counts[int(key)] += int(value)
        for key, value in items.value_counts().items():
            item_counts[int(key)] += int(value)

        action_counts['click'] += int(pd.to_numeric(chunk['click'], errors='coerce').fillna(0).gt(0).sum())
        for action in ['read', 'like', 'share', 'follow', 'favorite']:
            action_counts[action] += bool_count(chunk[action])

    column_profile = pd.DataFrame({
        'column': header,
        'dtype_observed': [dtypes[column] for column in header],
        'non_null_count': [non_null[column] for column in header],
        'missing_count': [n_rows - non_null[column] for column in header],
    })
    column_profile['missing_rate'] = column_profile['missing_count'] / n_rows

    n_users, n_items = len(user_set), len(item_set)
    # Tenrec 行记录是一次用户-物品行为；若存在同一 user-item 的重复行，下面密度会略高估。
    event_density = n_rows / (n_users * n_items)
    user_dist = pd.Series(user_counts, dtype='int64').describe(percentiles=[.5, .9, .95, .99]).to_dict()
    item_dist = pd.Series(item_counts, dtype='int64').describe(percentiles=[.5, .9, .95, .99]).to_dict()
    raw_profile = {
        'rows': int(n_rows),
        'unique_users': int(n_users),
        'unique_items': int(n_items),
        'event_density_upper_bound': float(event_density),
        'event_sparsity_lower_bound': float(1.0 - event_density),
        'positive_action_counts': {key: int(value) for key, value in action_counts.items()},
        'user_event_count_distribution': {key: float(value) for key, value in user_dist.items()},
        'item_event_count_distribution': {key: float(value) for key, value in item_dist.items()},
    }
    column_profile.to_csv(COLUMN_PROFILE_PATH, index=False)
    PROFILE_JSON_PATH.write_text(json.dumps(raw_profile, ensure_ascii=False, indent=2))

display(column_profile)
display(pd.DataFrame([raw_profile]).drop(columns=['user_event_count_distribution', 'item_event_count_distribution']))
print('用户交互次数分布：', raw_profile['user_event_count_distribution'])
print('物品交互次数分布：', raw_profile['item_event_count_distribution'])
print('正反馈行为量：', raw_profile['positive_action_counts'])

## 1. 分块抽样与行为权重

先扫一遍用户 ID，从全部用户中固定随机抽取 10 万；第二遍只写出这 10 万用户的必要字段。正反馈权重为：点击 1、阅读 1.5、点赞 3、分享 4、关注 4、收藏 5。

In [ ]:
SAMPLED_USERS_PATH = DATA_DIR / 'sampled_user_ids.npy'
EVENTS_PATH = DATA_DIR / 'qk_article_100k_events_v2.parquet'

if SAMPLED_USERS_PATH.exists():
    sampled_users = np.load(SAMPLED_USERS_PATH)
else:
    all_users = set()
    for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, usecols=['user_id'], chunksize=CHUNK_SIZE), desc='扫描用户'):
        all_users.update(chunk['user_id'].dropna().astype('int64').unique().tolist())
    all_users = np.fromiter(all_users, dtype=np.int64)
    assert len(all_users) >= N_USERS, f'仅发现 {len(all_users)} 个用户，少于设定的 {N_USERS}'
    sampled_users = np.random.default_rng(SEED).choice(all_users, size=N_USERS, replace=False)
    np.save(SAMPLED_USERS_PATH, np.sort(sampled_users))

sampled_user_set = set(sampled_users.tolist())
print(f'抽样用户数：{len(sampled_user_set):,}')

def as_bool(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.lower().isin(['1', '1.0', 'true', 't', 'yes'])

# category_first和category_second分别是文章的类别，“_第一”表示大类类别（如体育、娱乐、军事等），“_第二”表示细分类别（如 NBA、世界杯、科比等）
# 文章得分 1、文章得分 2、文章得分 3 分别表示文章的质量，采用不同的评分系统来进行衡量
required_columns = [
    'user_id', 'item_id', 'click', 'gender', 'age', 'category_first', 'category_second',
    'read', 'share', 'like', 'follow', 'favorite', 'read_percentage', 'read_time',
    'item_score1', 'item_score2', 'item_score3',
    'click_count', 'like_count', 'comment_count', 'exposure_count'
]


if not EVENTS_PATH.exists():
    event_parts = []
    source_order = 0
    reader = pd.read_csv(RAW_DATA_PATH, usecols=required_columns, chunksize=CHUNK_SIZE)
    for chunk in tqdm(reader, desc='过滤 10 万用户'):
        chunk = chunk[chunk['user_id'].isin(sampled_user_set)].copy()
        if chunk.empty:
            source_order += CHUNK_SIZE
            continue
        click = pd.to_numeric(chunk['click'], errors='coerce').fillna(0).gt(0)
        read = as_bool(chunk['read'])
        like = as_bool(chunk['like'])
        share = as_bool(chunk['share'])
        follow = as_bool(chunk['follow'])
        favorite = as_bool(chunk['favorite'])
        positive = click | read | like | share | follow | favorite
        chunk = chunk.loc[positive].copy()
        read_time = pd.to_numeric(chunk['read_time'], errors='coerce').fillna(0).clip(lower=0, upper=READ_TIME_CAP_SECONDS)
        read_pct = pd.to_numeric(chunk['read_percentage'], errors='coerce').fillna(0).clip(lower=0, upper=100) / 100.0
        weight = (click.loc[positive].astype(float) * 1.0 + read.loc[positive].astype(float) * 1.5 +
                  like.loc[positive].astype(float) * 3.0 + share.loc[positive].astype(float) * 4.0 +
                  follow.loc[positive].astype(float) * 4.0 + favorite.loc[positive].astype(float) * 5.0 +
                  READ_TIME_WEIGHT * np.log1p(read_time.loc[positive]) +
                  READ_PERCENTAGE_WEIGHT * read_pct.loc[positive])
        chunk['read_time'] = read_time.loc[positive]
        chunk['read_percentage'] = (read_pct.loc[positive] * 100).astype('float32')
        chunk['weight'] = weight.clip(lower=1.0)
        chunk['event_order'] = np.arange(source_order, source_order + len(chunk), dtype=np.int64)
        source_order += CHUNK_SIZE
        event_parts.append(chunk)
    events = pd.concat(event_parts, ignore_index=True)
    events.to_parquet(EVENTS_PATH, index=False)
    del event_parts
else:
    events = pd.read_parquet(EVENTS_PATH)

events['user_id'] = events['user_id'].astype('int64')
events['item_id'] = events['item_id'].astype('int64')
print(events.shape)
events.head()

## 2. 按用户时间顺序切分

对每个用户保留最后一次正反馈为测试、倒数第二次为验证，其余为训练。重复的 user-item 会按最大行为权重聚合到训练矩阵；测试目标仍按最后行为定义。

In [ ]:
TRAIN_PATH = DATA_DIR / 'train_interactions_v2.parquet'
VAL_PATH = DATA_DIR / 'val_interactions_v2.parquet'
TEST_PATH = DATA_DIR / 'test_interactions_v2.parquet'
ITEM_META_PATH = DATA_DIR / 'item_metadata_v2.parquet'
USER_META_PATH = DATA_DIR / 'user_metadata_v2.parquet'

if not TRAIN_PATH.exists():
    events = events.sort_values(['user_id', 'event_order'], kind='stable').reset_index(drop=True)
    event_count = events.groupby('user_id')['item_id'].transform('size')
    eligible = events.loc[event_count >= MIN_POSITIVE_EVENTS].copy()
    eligible['rank_from_end'] = eligible.groupby('user_id').cumcount(ascending=False)
    test_df = eligible[eligible['rank_from_end'] == 0].copy()
    val_df = eligible[eligible['rank_from_end'] == 1].copy()
    train_df = eligible[eligible['rank_from_end'] >= 2].copy()

    # 训练矩阵中将重复交互按最大偏好权重合并。
    train_df = (train_df.sort_values('event_order')
                .groupby(['user_id', 'item_id'], as_index=False)
                .agg(weight=('weight', 'max'),
                     category_first=('category_first', 'last'),
                     category_second=('category_second', 'last'),
                     gender=('gender', 'last'), age=('age', 'last'),
                     read_time=('read_time', 'max'), read_percentage=('read_percentage', 'max'),
                     item_score1=('item_score1', 'last'), item_score2=('item_score2', 'last'), item_score3=('item_score3', 'last'),
                     click_count=('click_count', 'last'), like_count=('like_count', 'last'),
                     comment_count=('comment_count', 'last'), exposure_count=('exposure_count', 'last')))

    # 元数据仅由训练集构建，避免类别/质量特征通过测试行为泄漏。
    item_meta = (train_df
                 .groupby('item_id', as_index=False)
                 .agg(category_first=('category_first', 'last'),
                      category_second=('category_second', 'last'),
                      item_score1=('item_score1', 'last'), item_score2=('item_score2', 'last'), item_score3=('item_score3', 'last'),
                      click_count=('click_count', 'last'), like_count=('like_count', 'last'),
                      comment_count=('comment_count', 'last'), exposure_count=('exposure_count', 'last')))
    user_meta = (train_df
                 .groupby('user_id', as_index=False)
                 .agg(gender=('gender', 'last'), age=('age', 'last'),
                      mean_read_time=('read_time', 'mean'), mean_read_percentage=('read_percentage', 'mean')))

    train_df.to_parquet(TRAIN_PATH, index=False)
    val_df[['user_id', 'item_id']].to_parquet(VAL_PATH, index=False)
    test_df[['user_id', 'item_id']].to_parquet(TEST_PATH, index=False)
    item_meta.to_parquet(ITEM_META_PATH, index=False)
    user_meta.to_parquet(USER_META_PATH, index=False)
else:
    train_df = pd.read_parquet(TRAIN_PATH)
    val_df = pd.read_parquet(VAL_PATH)
    test_df = pd.read_parquet(TEST_PATH)
    item_meta = pd.read_parquet(ITEM_META_PATH)
    user_meta = pd.read_parquet(USER_META_PATH)

print({'train': train_df.shape, 'val': val_df.shape, 'test': test_df.shape, 'items': len(item_meta)})
train_df.head()

In [ ]:
# 统一的 ID 编码、训练矩阵、已看集合和评估用户。
user_ids = np.sort(train_df['user_id'].unique())
item_ids = np.sort(train_df['item_id'].unique())
user_to_idx = {value: idx for idx, value in enumerate(user_ids)}
item_to_idx = {value: idx for idx, value in enumerate(item_ids)}
idx_to_user = user_ids
idx_to_item = item_ids

rows = train_df['user_id'].map(user_to_idx).to_numpy()
cols = train_df['item_id'].map(item_to_idx).to_numpy()
values = train_df['weight'].astype('float32').to_numpy()
user_item = sparse.csr_matrix((values, (rows, cols)), shape=(len(user_ids), len(item_ids)), dtype=np.float32)
seen_by_user = {u: set(frame['item_id'].tolist()) for u, frame in train_df.groupby('user_id')}
test_truth = dict(zip(test_df['user_id'], test_df['item_id']))
eval_user_ids = np.array(sorted(test_truth), dtype=np.int64)
eval_user_ids = np.random.default_rng(SEED).choice(eval_user_ids, size=min(EVAL_USERS, len(eval_user_ids)), replace=False)

def to_candidate_frame(rows, source):
    return pd.DataFrame(rows, columns=['user_id', 'item_id', 'score']).assign(source=source)

def recall_at_k(candidates: pd.DataFrame, truth: dict[int, int], k: int = RECALL_K) -> float:
    ranked = candidates.sort_values(['user_id', 'score'], ascending=[True, False]).groupby('user_id').head(k)
    hit = ranked.apply(lambda x: truth.get(int(x.user_id)) == int(x.item_id), axis=1).groupby(ranked['user_id']).any()
    users = [u for u in truth if u in set(ranked['user_id'])]
    return float(hit.reindex(users, fill_value=False).mean()) if users else 0.0

print(user_item.shape, user_item.nnz)

## 3. ItemCF 与 UserCF

ItemCF 使用 BM25 加权的 item-item 相似度；UserCF 使用余弦用户相似度。两者都只依赖训练交互，且过滤已看内容。

In [ ]:
item_cf = BM25Recommender(K=SIMILAR_K)
item_cf.fit(user_item.T.tocsr(), show_progress=True)

user_cf = CosineRecommender(K=SIMILAR_K)
user_cf.fit(user_item.tocsr(), show_progress=True)

def itemcf_candidates(user_id: int, n: int = RECALL_K):
    uidx = user_to_idx[user_id]
    item_idx, scores = item_cf.recommend(uidx, user_item, N=n, filter_already_liked_items=True)
    return [(user_id, int(idx_to_item[i]), float(s)) for i, s in zip(item_idx, scores)]

def usercf_candidates(user_id: int, n: int = RECALL_K):
    uidx = user_to_idx[user_id]
    neighbour_idx, neighbour_scores = user_cf.similar_items(uidx, N=SIMILAR_K + 1)
    scores = defaultdict(float)
    seen = seen_by_user[user_id]
    for nidx, similarity in zip(neighbour_idx, neighbour_scores):
        if int(nidx) == uidx:
            continue
        start, end = user_item.indptr[int(nidx)], user_item.indptr[int(nidx) + 1]
        for item_idx, weight in zip(user_item.indices[start:end], user_item.data[start:end]):
            item_id = int(idx_to_item[item_idx])
            if item_id not in seen:
                scores[item_id] += float(similarity) * float(weight)
    return [(user_id, item_id, score) for item_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]]

itemcf_rows, usercf_rows = [], []
for user_id in tqdm(eval_user_ids, desc='生成 CF 候选'):
    user_id = int(user_id)
    itemcf_rows.extend(itemcf_candidates(user_id))
    usercf_rows.extend(usercf_candidates(user_id))

itemcf_result = to_candidate_frame(itemcf_rows, 'itemcf')
usercf_result = to_candidate_frame(usercf_rows, 'usercf')
itemcf_result.to_parquet(OUTPUT_DIR / 'candidates_itemcf.parquet', index=False)
usercf_result.to_parquet(OUTPUT_DIR / 'candidates_usercf.parquet', index=False)
print('ItemCF Recall@%d:' % RECALL_K, recall_at_k(itemcf_result, test_truth))
print('UserCF Recall@%d:' % RECALL_K, recall_at_k(usercf_result, test_truth))

## 4. 标签召回

用训练集统计用户对二级类目的偏好，给每个类目建立加权热门内容池；按用户最偏好的类目取候选。

In [ ]:
TAG_TOP_ITEMS = 500
TAG_PER_USER = 5
tag_col = 'category_second'
tag_train = train_df.dropna(subset=[tag_col]).copy()
tag_train[tag_col] = tag_train[tag_col].astype('int64')

popular_by_tag = (tag_train.groupby([tag_col, 'item_id'], as_index=False)['weight'].sum()
                  .sort_values([tag_col, 'weight'], ascending=[True, False])
                  .groupby(tag_col).head(TAG_TOP_ITEMS))
items_by_tag = {tag: list(zip(frame['item_id'].astype(int), frame['weight'].astype(float)))
                for tag, frame in popular_by_tag.groupby(tag_col)}
user_tag_pref = (tag_train.groupby(['user_id', tag_col], as_index=False)['weight'].sum()
                 .sort_values(['user_id', 'weight'], ascending=[True, False])
                 .groupby('user_id').head(TAG_PER_USER))

def tag_candidates(user_id: int, n: int = RECALL_K):
    prefs = user_tag_pref[user_tag_pref['user_id'] == user_id]
    seen, scores = seen_by_user[user_id], defaultdict(float)
    for _, row in prefs.iterrows():
        for item_id, item_popularity in items_by_tag.get(int(row[tag_col]), []):
            if item_id not in seen:
                scores[item_id] += float(row['weight']) * item_popularity
    return [(user_id, item_id, score) for item_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]]

tag_rows = []
for user_id in tqdm(eval_user_ids, desc='生成标签候选'):
    tag_rows.extend(tag_candidates(int(user_id)))
tag_result = to_candidate_frame(tag_rows, 'tag')
tag_result.to_parquet(OUTPUT_DIR / 'candidates_tag.parquet', index=False)
print('标签召回 Recall@%d:' % RECALL_K, recall_at_k(tag_result, test_truth))

## 4.5 热门召回

默认 `train_only` 根据训练期内的交互数和加权行为强度计算热门度，适合严格离线评测。若把 `POPULARITY_MODE` 改为 `provided_item_stats`，则会使用数据集的 `click_count / like_count / comment_count / exposure_count`：点击、点赞、评论和曝光越多，且点击率越高，文章越热门。后者可能含有测试期的全局累计信息，只应用于特征对比，不应报告为严格离线指标。

In [ ]:
popularity = (train_df.groupby('item_id', as_index=False)
              .agg(train_event_count=('user_id', 'size'), train_weight_sum=('weight', 'sum'))
              .merge(item_meta[['item_id', 'click_count', 'like_count', 'comment_count', 'exposure_count']], on='item_id', how='left'))

if POPULARITY_MODE == 'train_only':
    popularity['popularity_score'] = np.log1p(popularity['train_event_count']) + np.log1p(popularity['train_weight_sum'])
elif POPULARITY_MODE == 'provided_item_stats':
    for column in ['click_count', 'like_count', 'comment_count', 'exposure_count']:
        popularity[column] = pd.to_numeric(popularity[column], errors='coerce').fillna(0).clip(lower=0)
    ctr_proxy = popularity['click_count'] / (popularity['exposure_count'] + 1.0)
    popularity['popularity_score'] = (np.log1p(popularity['click_count']) +
                                       2.0 * np.log1p(popularity['like_count']) +
                                       0.5 * np.log1p(popularity['comment_count']) +
                                       0.25 * np.log1p(popularity['exposure_count']) +
                                       4.0 * ctr_proxy)
else:
    raise ValueError("POPULARITY_MODE 只能是 'train_only' 或 'provided_item_stats'")

popular_items = popularity.sort_values('popularity_score', ascending=False)[['item_id', 'popularity_score']].itertuples(index=False, name=None)
popular_items = list(popular_items)

def popularity_candidates(user_id: int, n: int = RECALL_K):
    seen, rows = seen_by_user[user_id], []
    for item_id, score in popular_items:
        if int(item_id) not in seen:
            rows.append((user_id, int(item_id), float(score)))
            if len(rows) == n:
                break
    return rows

popularity_rows = []
for user_id in tqdm(eval_user_ids, desc='生成热门候选'):
    popularity_rows.extend(popularity_candidates(int(user_id)))
popularity_result = to_candidate_frame(popularity_rows, 'popularity')
popularity_result.to_parquet(OUTPUT_DIR / f'candidates_popularity_{POPULARITY_MODE}.parquet', index=False)
print(f'热门召回（{POPULARITY_MODE}） Recall@{RECALL_K}:', recall_at_k(popularity_result, test_truth))

## 5. 双塔召回

用户塔：user ID + 性别 + 年龄 + 历史平均阅读时长/完成度；物品塔：item ID + 一级/二级标签 + 三个文章质量分。采用 BPR 负采样训练。为了控制成本，下面只对 `EVAL_USERS` 做暴力内积召回；正式服务阶段可将物品向量建立 FAISS/Milvus 索引。

In [ ]:
LATENT_DIM = 64
BATCH_SIZE = 4096
EPOCHS = 5
LEARNING_RATE = 2e-3

def make_code_map(series: pd.Series):
    values = sorted(series.dropna().astype(str).unique().tolist())
    return {value: index + 1 for index, value in enumerate(values)}

gender_map = make_code_map(user_meta['gender'])
age_map = make_code_map(user_meta['age'])
cat1_map = make_code_map(item_meta['category_first'])
cat2_map = make_code_map(item_meta['category_second'])

user_feature_frame = pd.DataFrame({'user_id': user_ids}).merge(user_meta, on='user_id', how='left')
item_feature_frame = pd.DataFrame({'item_id': item_ids}).merge(item_meta, on='item_id', how='left')
user_gender = user_feature_frame['gender'].astype(str).map(gender_map).fillna(0).astype('int64').to_numpy()
user_age = user_feature_frame['age'].astype(str).map(age_map).fillna(0).astype('int64').to_numpy()
item_cat1 = item_feature_frame['category_first'].astype(str).map(cat1_map).fillna(0).astype('int64').to_numpy()
item_cat2 = item_feature_frame['category_second'].astype(str).map(cat2_map).fillna(0).astype('int64').to_numpy()

def clipped_standardize(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    values = frame[columns].apply(pd.to_numeric, errors='coerce').fillna(0.0).to_numpy(dtype=np.float32)
    lower, upper = np.quantile(values, 0.01, axis=0), np.quantile(values, 0.99, axis=0)
    values = np.clip(values, lower, upper)
    return ((values - values.mean(axis=0)) / (values.std(axis=0) + 1e-6)).astype(np.float32)

# read_time/read_percentage 只来自用户过去的训练行为；item_score 是候选文章推荐前可获得的质量信息。
user_engagement = clipped_standardize(user_feature_frame, ['mean_read_time', 'mean_read_percentage'])
item_quality = clipped_standardize(item_feature_frame, ['item_score1', 'item_score2', 'item_score3'])

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_id_emb = nn.Embedding(len(user_ids), LATENT_DIM)
        self.gender_emb = nn.Embedding(len(gender_map) + 1, 8)
        self.age_emb = nn.Embedding(len(age_map) + 1, 8)
        self.item_id_emb = nn.Embedding(len(item_ids), LATENT_DIM)
        self.cat1_emb = nn.Embedding(len(cat1_map) + 1, 16)
        self.cat2_emb = nn.Embedding(len(cat2_map) + 1, 16)
        self.user_mlp = nn.Sequential(nn.Linear(LATENT_DIM + 16 + 2, 128), nn.ReLU(), nn.Linear(128, LATENT_DIM))
        self.item_mlp = nn.Sequential(nn.Linear(LATENT_DIM + 32 + 3, 128), nn.ReLU(), nn.Linear(128, LATENT_DIM))
    def user_tower(self, uid, gender, age, engagement):
        x = torch.cat([self.user_id_emb(uid), self.gender_emb(gender), self.age_emb(age), engagement], dim=-1)
        return nn.functional.normalize(self.user_mlp(x), dim=-1)
    def item_tower(self, iid, cat1, cat2, quality):
        x = torch.cat([self.item_id_emb(iid), self.cat1_emb(cat1), self.cat2_emb(cat2), quality], dim=-1)
        return nn.functional.normalize(self.item_mlp(x), dim=-1)

train_uid = train_df['user_id'].map(user_to_idx).to_numpy(dtype=np.int64)
train_iid = train_df['item_id'].map(item_to_idx).to_numpy(dtype=np.int64)
train_w = np.log1p(train_df['weight'].to_numpy(dtype=np.float32))
loader = DataLoader(TensorDataset(torch.from_numpy(train_uid), torch.from_numpy(train_iid), torch.from_numpy(train_w)),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())

model = TwoTower().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
user_gender_t = torch.tensor(user_gender, device=DEVICE)
user_age_t = torch.tensor(user_age, device=DEVICE)
item_cat1_t = torch.tensor(item_cat1, device=DEVICE)
item_cat2_t = torch.tensor(item_cat2, device=DEVICE)
user_engagement_t = torch.tensor(user_engagement, device=DEVICE)
item_quality_t = torch.tensor(item_quality, device=DEVICE)

for epoch in range(1, EPOCHS + 1):
    model.train(); total_loss = 0.0
    for uid, pos_iid, sample_weight in tqdm(loader, desc=f'双塔 epoch {epoch}', leave=False):
        uid, pos_iid, sample_weight = uid.to(DEVICE), pos_iid.to(DEVICE), sample_weight.to(DEVICE)
        neg_iid = torch.randint(len(item_ids), size=pos_iid.shape, device=DEVICE)
        neg_iid = torch.where(neg_iid == pos_iid, (neg_iid + 1) % len(item_ids), neg_iid)
        user_vec = model.user_tower(uid, user_gender_t[uid], user_age_t[uid], user_engagement_t[uid])
        pos_vec = model.item_tower(pos_iid, item_cat1_t[pos_iid], item_cat2_t[pos_iid], item_quality_t[pos_iid])
        neg_vec = model.item_tower(neg_iid, item_cat1_t[neg_iid], item_cat2_t[neg_iid], item_quality_t[neg_iid])
        bpr = -nn.functional.logsigmoid((user_vec * pos_vec).sum(-1) - (user_vec * neg_vec).sum(-1))
        loss = (bpr * sample_weight).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += float(loss.detach()) * len(uid)
    print(f'epoch={epoch}, weighted_bpr_loss={total_loss / len(train_df):.5f}')

torch.save({'state_dict': model.state_dict(), 'user_ids': user_ids, 'item_ids': item_ids}, MODEL_DIR / 'two_tower.pt')

In [ ]:
@torch.no_grad()
def two_tower_candidates(user_id: int, n: int = RECALL_K):
    model.eval()
    all_item_idx = torch.arange(len(item_ids), device=DEVICE)
    item_vec = model.item_tower(all_item_idx, item_cat1_t, item_cat2_t, item_quality_t)
    uidx = user_to_idx[user_id]
    uid = torch.tensor([uidx], device=DEVICE)
    user_vec = model.user_tower(uid, user_gender_t[uid], user_age_t[uid], user_engagement_t[uid])
    scores = (user_vec @ item_vec.T).squeeze(0)
    seen_idx = [item_to_idx[i] for i in seen_by_user[user_id] if i in item_to_idx]
    scores[torch.tensor(seen_idx, device=DEVICE)] = -torch.inf
    top_scores, top_idx = torch.topk(scores, k=n)
    return [(user_id, int(idx_to_item[i]), float(s)) for i, s in zip(top_idx.cpu().numpy(), top_scores.cpu().numpy())]

two_tower_rows = []
for user_id in tqdm(eval_user_ids, desc='生成双塔候选'):
    two_tower_rows.extend(two_tower_candidates(int(user_id)))
two_tower_result = to_candidate_frame(two_tower_rows, 'two_tower')
two_tower_result.to_parquet(OUTPUT_DIR / 'candidates_two_tower.parquet', index=False)
print('双塔 Recall@%d:' % RECALL_K, recall_at_k(two_tower_result, test_truth))

## 6. 多路合并与结果保存

这里先做去重并保留来源，不做跨路分数归一化；后续粗排会学习每一路的分数、排名和来源等特征。

In [ ]:
all_candidates = pd.concat([itemcf_result, usercf_result, tag_result, popularity_result, two_tower_result], ignore_index=True)
all_candidates['rank_in_source'] = all_candidates.groupby(['source', 'user_id'])['score'].rank(method='first', ascending=False)
merged_candidates = (all_candidates.sort_values(['user_id', 'source', 'score'], ascending=[True, True, False])
                     .groupby(['user_id', 'item_id'], as_index=False)
                     .agg(sources=('source', lambda x: '|'.join(sorted(set(x)))),
                          best_rank=('rank_in_source', 'min'),
                          max_score=('score', 'max')))
merged_candidates.to_parquet(OUTPUT_DIR / 'candidates_merged.parquet', index=False)

print('候选输出目录：', OUTPUT_DIR)
print('各路候选条数：')
display(all_candidates.groupby('source').size().rename('rows').to_frame())
print('去重后的候选数：', len(merged_candidates))
display(merged_candidates.head(10))